# 机械手指运动学 — 逐步构建

---

## 平面档案

| 面名 | 构造方式 | 关键参数 |
|------|----------|----------|
| `finger` | Z轴起点 → Y方向轴 → 绕轴旋转 φ | `z_offset=30, L_shaft=5, φ=0°` |

---

## 第一步：面的构造

1. **起点** (`base`)：Z 轴上，由 `z_offset` 定位
2. **轴** (`shaft`)：从 base 沿 Y 方向拉出，长度 `L_shaft`，末端为 `shaft_end`
3. **面**：包含这条轴，垂直于 ZX 平面，绕轴旋转角 `φ` 定位
   - φ=0 时面朝 +Z 方向
   - φ>0 面朝 +X 方向偏转

In [ ]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider, Checkbox

# ============================================================
# 面注册表
# ============================================================

planes = {}

def add_plane(name, description, params, points_fn):
    planes[name] = dict(
        description=description,
        params=dict(params),
        points_fn=points_fn,
    )

def get_points(name, **overrides):
    p = {**planes[name]['params'], **overrides}
    return planes[name]['points_fn'](**p)

def show_planes():
    for name, pl in planes.items():
        pts = pl['points_fn'](**pl['params'])
        print(f"[{name}] — {pl['description']}")
        print(f"  params : {list(pl['params'].keys())}")
        print(f"  points : {list(pts.keys())}")

# ============================================================
# 面：finger
# 构造：Z轴起点 → Y方向轴 → 绕轴旋转 φ
# ============================================================

def finger_points(z_offset, L_shaft, phi):
    p = np.radians(phi)
    base      = np.array([0.0, 0.0, z_offset])
    shaft_end = np.array([0.0, L_shaft, z_offset])
    r_dir = np.array([np.sin(p), 0.0, np.cos(p)])
    return {
        'base':      base,
        'shaft_end': shaft_end,
        'r_dir':     r_dir,
    }

add_plane(
    name='finger',
    description='Z轴起点 → Y轴 → 绕轴旋转',
    params=dict(z_offset=30.0, L_shaft=5.0, phi=0.0),
    points_fn=finger_points,
)

show_planes()

# ============================================================
# 可视化
# ============================================================

OPACITY_FADED = 0.2

def draw_finger(show_finger_plane=True, z_offset=30.0, L_shaft=5.0, phi=0.0):

    pts = get_points('finger', z_offset=z_offset, L_shaft=L_shaft, phi=phi)
    alpha = 1.0 if show_finger_plane else OPACITY_FADED

    base      = pts['base']
    shaft_end = pts['shaft_end']
    r_dir     = pts['r_dir']
    y_dir     = np.array([0.0, 1.0, 0.0])

    fig = go.Figure()
    span = max(z_offset + L_shaft + 20, 50)

    # --- ZX 参考平面（淡灰底） ---
    fig.add_trace(go.Surface(
        x=[[-span, span], [-span, span]],
        y=[[0, 0], [0, 0]],
        z=[[-span, span], [-span, span]],
        opacity=0.04, colorscale='greys', showscale=False,
        hoverinfo='skip', name='ZX ref',
    ))

    # --- finger 面：网格线 ---
    if show_finger_plane:
        half = z_offset + 15
        for s in np.linspace(-half, half, 9):
            p1 = base + s * y_dir - half * r_dir
            p2 = base + s * y_dir + half * r_dir
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                mode='lines', line=dict(color='royalblue', width=1),
                opacity=0.12, hoverinfo='skip', showlegend=False,
            ))
            p1 = base - half * y_dir + s * r_dir
            p2 = base + half * y_dir + s * r_dir
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                mode='lines', line=dict(color='royalblue', width=1),
                opacity=0.12, hoverinfo='skip', showlegend=False,
            ))

    # --- 轴线：base → shaft_end ---
    fig.add_trace(go.Scatter3d(
        x=[base[0], shaft_end[0]],
        y=[base[1], shaft_end[1]],
        z=[base[2], shaft_end[2]],
        mode='lines+markers+text',
        line=dict(color='darkorange', width=5),
        marker=dict(size=[10, 8], color=['yellow', 'darkorange'],
                    symbol=['diamond', 'circle'],
                    line=dict(color='orange', width=2)),
        text=['base', 'shaft_end'], textposition='top center',
        hovertemplate='%{text}<br>X=%{x:.1f} Y=%{y:.1f} Z=%{z:.1f}<extra></extra>',
        name='shaft',
        opacity=alpha,
    ))

    # --- r_dir 方向指示（从 base 出发） ---
    arrow_len = 15
    arrow_tip = base + arrow_len * r_dir
    fig.add_trace(go.Scatter3d(
        x=[base[0], arrow_tip[0]],
        y=[base[1], arrow_tip[1]],
        z=[base[2], arrow_tip[2]],
        mode='lines+text',
        line=dict(color='green', width=3, dash='dash'),
        text=[None, 'r_dir'], textposition='top center',
        hoverinfo='skip',
        name=f'r_dir (φ={phi:.0f}°)',
        opacity=alpha,
    ))

    # --- 原点标记 ---
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0], mode='markers+text',
        marker=dict(size=8, color='black', symbol='cross'),
        text=['O(0,0,0)'], textposition='bottom center',
        name='origin',
    ))

    # --- Z 轴参考线 ---
    fig.add_trace(go.Scatter3d(
        x=[0, 0], y=[0, 0], z=[0, span],
        mode='lines', line=dict(color='gray', width=1, dash='dot'),
        hoverinfo='skip', showlegend=False, opacity=0.4,
    ))

    # --- 布局：以原点为锚 ---
    fig.update_layout(
        title=f'Plane: z_offset={z_offset:.0f}  L_shaft={L_shaft:.0f}  φ={phi:.0f}°',
        scene=dict(
            xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
            xaxis=dict(range=[-span, span]),
            yaxis=dict(range=[-span, span]),
            zaxis=dict(range=[0, span * 2]),
            aspectmode='cube',
            camera=dict(
                center=dict(x=0, y=0, z=-0.2),
                eye=dict(x=0, y=-1.8, z=0.5),
                up=dict(x=0, y=0, z=1),
            ),
        ),
        width=800, height=700,
        showlegend=True,
    )
    fig.show()

# ============================================================
# 交互控件
# ============================================================

interact(draw_finger,
         show_finger_plane=Checkbox(value=True, description='show finger plane'),
         z_offset =FloatSlider(value=30, min=0,  max=80, step=1,   description='z_offset'),
         L_shaft  =FloatSlider(value=5,  min=1,  max=30, step=1,   description='L_shaft'),
         phi      =FloatSlider(value=0,  min=-90,max=90, step=1,   description='φ (°)'),
);

In [9]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# 系统参数（所有长度单位：mm）
# ============================================================

# 世界坐标系中的固定点
z0 = 5.0          # CPS 和 CPE 的 Z 坐标（相同高度）
L_cp = 10.0       # cp 线长度：CPS -> CPE 沿 +X 方向

CPS = np.array([0.0, 0.0, z0])   # cp-start (在 Z 轴上)
CPE = np.array([L_cp, 0.0, z0])  # cp-end (固定点，也是旋转轴上的点)

# rotate-sub-system 内部参数
L_A = 7.0         # OA 长度（CPE → A）
L_B = 14.0        # 经过 A 的垂直线段长度（对称于 ZX 平面）

# 角度参数
alpha_deg = 45    # OA 与局部 +Z 轴的夹角（在 ZX 平面内）
theta_deg = 30    # 子系统绕 X 轴（经过 CPE）的旋转角

alpha = np.deg2rad(alpha_deg)
theta = np.deg2rad(theta_deg)

# ============================================================
# 辅助函数
# ============================================================

def rot_x_matrix(angle_rad):
    """绕 X 轴旋转的 3x3 矩阵（右手定则）"""
    c = np.cos(angle_rad)
    s = np.sin(angle_rad)
    return np.array([
        [1, 0, 0],
        [0, c, -s],
        [0, s, c]
    ])

def rotate_around_X_through_point(points, point_on_axis, angle_rad):
    """
    绕经过 point_on_axis、方向为 X 轴的直线旋转
    points: (N, 3) 点集
    point_on_axis: 轴上一点（例如 CPE）
    """
    R = rot_x_matrix(angle_rad)
    points_rot = np.zeros_like(points)
    for i, p in enumerate(points):
        p_rel = p - point_on_axis
        p_rot_rel = R @ p_rel
        points_rot[i] = point_on_axis + p_rot_rel
    return points_rot

def get_local_axes(origin, theta_rad):
    """
    获取局部坐标系的三个轴方向（世界坐标中的向量）
    初始局部坐标系与世界对齐，然后绕经过 origin 的 X 轴旋转 theta
    返回: (X_axis, Y_axis, Z_axis) 每个都是 (3,) 向量（已归一化）
    """
    # 初始轴（与世界对齐）
    X_init = np.array([1, 0, 0])
    Y_init = np.array([0, 1, 0])
    Z_init = np.array([0, 0, 1])
    
    # 绕 X 轴旋转 theta
    R = rot_x_matrix(theta_rad)
    
    X_axis = R @ X_init
    Y_axis = R @ Y_init
    Z_axis = R @ Z_init
    
    return X_axis, Y_axis, Z_axis

# ============================================================
# 1. 构建局部坐标系中的点（初始状态，与世界对齐）
# ============================================================

# 点 A（在局部 ZX 平面内，Y=0）
xA = L_A * np.sin(alpha)
zA = L_A * np.cos(alpha)
A_local = np.array([xA, 0.0, zA])

# 垂直线段端点（经过 A，平行于 Y 轴，对称于 ZX 平面）
half_LB = L_B / 2.0
A_plus_local  = np.array([xA,  half_LB, zA])
A_minus_local = np.array([xA, -half_LB, zA])

# 所有局部点（相对于 CPE）
local_points = np.array([A_local, A_plus_local, A_minus_local])

# ============================================================
# 2. 平移到世界坐标（加上 CPE 的平移）
# ============================================================

points_world_init = local_points + CPE   # 初始世界坐标（未旋转）

# ============================================================
# 3. 应用旋转（绕经过 CPE 的 X 轴）
# ============================================================

points_rot = rotate_around_X_through_point(points_world_init, CPE, theta)

A_rot     = points_rot[0]
A_plus_rot  = points_rot[1]
A_minus_rot = points_rot[2]

# CPE 自身不变（因为它在轴上）
CPE_rot = CPE.copy()

# ============================================================
# 4. 获取局部坐标系的轴（世界坐标中的方向）
# ============================================================

X_local, Y_local, Z_local = get_local_axes(CPE, theta)

# 用于显示坐标轴的线段长度
axis_len = 8.0

# 局部 X' 轴（红色）：从 CPE 指向 X_local 方向
X_axis_line = go.Scatter3d(
    x=[CPE[0], CPE[0] + X_local[0] * axis_len],
    y=[CPE[1], CPE[1] + X_local[1] * axis_len],
    z=[CPE[2], CPE[2] + X_local[2] * axis_len],
    mode='lines',
    line=dict(color='red', width=3),
    name='局部 X\' 轴'
)

# 局部 Y' 轴（绿色）
Y_axis_line = go.Scatter3d(
    x=[CPE[0], CPE[0] + Y_local[0] * axis_len],
    y=[CPE[1], CPE[1] + Y_local[1] * axis_len],
    z=[CPE[2], CPE[2] + Y_local[2] * axis_len],
    mode='lines',
    line=dict(color='green', width=3),
    name='局部 Y\' 轴'
)

# 局部 Z' 轴（蓝色）
Z_axis_line = go.Scatter3d(
    x=[CPE[0], CPE[0] + Z_local[0] * axis_len],
    y=[CPE[1], CPE[1] + Z_local[1] * axis_len],
    z=[CPE[2], CPE[2] + Z_local[2] * axis_len],
    mode='lines',
    line=dict(color='blue', width=3),
    name='局部 Z\' 轴'
)

# ============================================================
# 5. 构建 Plotly 图形
# ============================================================

# ---- 世界坐标轴 ----
world_axis_lines = []
world_axis_colors = ['red', 'green', 'blue']
world_axis_labels = ['世界 X', '世界 Y', '世界 Z']
world_axis_ends = np.array([
    [15, 0, 0],
    [0, 15, 0],
    [0, 0, 15]
])
for i, end in enumerate(world_axis_ends):
    world_axis_lines.append(go.Scatter3d(
        x=[0, end[0]], y=[0, end[1]], z=[0, end[2]],
        mode='lines',
        line=dict(color=world_axis_colors[i], width=2, dash='dot'),
        name=f'{world_axis_labels[i]}'
    ))

# ---- CPS 和 CPE（固定） ----
fixed_points = go.Scatter3d(
    x=[CPS[0], CPE[0]],
    y=[CPS[1], CPE[1]],
    z=[CPS[2], CPE[2]],
    mode='markers+text',
    marker=dict(size=8, color=['green', 'blue']),
    text=['CPS (cp-start)', 'CPE (cp-end)'],
    textposition='top center',
    name='固定点'
)

# ---- cp 线（CPS → CPE） ----
cp_line = go.Scatter3d(
    x=[CPS[0], CPE[0]],
    y=[CPS[1], CPE[1]],
    z=[CPS[2], CPE[2]],
    mode='lines',
    line=dict(color='gray', width=4, dash='dot'),
    name='cp 线 (固定)'
)

# ---- OA 线段（CPE → A） ----
oa_line = go.Scatter3d(
    x=[CPE[0], A_rot[0]],
    y=[CPE[1], A_rot[1]],
    z=[CPE[2], A_rot[2]],
    mode='lines',
    line=dict(color='orange', width=4),
    name=f'OA (L_A={L_A}mm, α={alpha_deg}°)'
)

# ---- 经过 A 的垂直线段 ----
vert_line = go.Scatter3d(
    x=[A_plus_rot[0], A_minus_rot[0]],
    y=[A_plus_rot[1], A_minus_rot[1]],
    z=[A_plus_rot[2], A_minus_rot[2]],
    mode='lines',
    line=dict(color='purple', width=5),
    name=f'垂直线段 (L_B={L_B}mm)'
)

# ---- 关键点标记（A, A+, A-） ----
points_markers = go.Scatter3d(
    x=[A_rot[0], A_plus_rot[0], A_minus_rot[0]],
    y=[A_rot[1], A_plus_rot[1], A_minus_rot[1]],
    z=[A_rot[2], A_plus_rot[2], A_minus_rot[2]],
    mode='markers+text',
    marker=dict(size=6, color=['blue', 'red', 'red']),
    text=['A', 'A+', 'A-'],
    textposition='top center',
    name='子系统内部点'
)

# ---- 旋转轴可视化（经过 CPE 且沿 X 方向） ----
axis_length = 20
axis_start = CPE + np.array([-axis_length, 0, 0])
axis_end   = CPE + np.array([ axis_length, 0, 0])
rot_axis_line = go.Scatter3d(
    x=[axis_start[0], axis_end[0]],
    y=[axis_start[1], axis_end[1]],
    z=[axis_start[2], axis_end[2]],
    mode='lines',
    line=dict(color='cyan', width=3, dash='dash'),
    name='旋转轴 (过 CPE, 沿 X 方向)'
)

# ============================================================
# 6. 绘图
# ============================================================

fig = go.Figure(data=[
    *world_axis_lines,
    fixed_points,
    cp_line,
    oa_line,
    vert_line,
    points_markers,
    rot_axis_line,
    X_axis_line,
    Y_axis_line,
    Z_axis_line
])

fig.update_layout(
    title=dict(
        text=f'完整系统 + 子系统局部坐标系<br>'
             f'CPS → CPE (L_cp={L_cp}mm) | α={alpha_deg}°, θ={theta_deg}° '
             f'(绕经过 CPE 的 X 轴)',
        font=dict(size=14)
    ),
    scene=dict(
        xaxis_title='X (mm)',
        yaxis_title='Y (mm)',
        zaxis_title='Z (mm)',
        aspectmode='data',
        camera=dict(eye=dict(x=1.8, y=1.8, z=1.5))
    ),
    width=1000,
    height=800,
    showlegend=True
)

fig.show()

# ============================================================
# 7. 打印关键信息
# ============================================================
print("=" * 60)
print("系统参数：")
print(f"  CPS = {CPS}")
print(f"  CPE = {CPE} (固定，在旋转轴上)")
print(f"  L_cp = {L_cp} mm")
print(f"  L_A = {L_A} mm")
print(f"  L_B = {L_B} mm")
print(f"  α = {alpha_deg}°")
print(f"  θ = {theta_deg}°")
print("=" * 60)
print("旋转后的点坐标：")
print(f"  A   = {A_rot}")
print(f"  A+  = {A_plus_rot}")
print(f"  A-  = {A_minus_rot}")
print("=" * 60)
print("局部坐标系轴方向（在世界坐标中）：")
print(f"  X' = {X_local}")
print(f"  Y' = {Y_local}")
print(f"  Z' = {Z_local}")
print("=" * 60)
print("注意：")
print("  - 世界坐标轴为虚线（红X, 绿Y, 蓝Z）")
print("  - 子系统局部坐标轴为实线（红X', 绿Y', 蓝Z'）")
print("  - 旋转轴（青色虚线）经过 CPE，方向沿 X 轴")
print("  - OA 在局部 ZX' 平面内（Y'=0）")

系统参数：
  CPS = [0. 0. 5.]
  CPE = [10.  0.  5.] (固定，在旋转轴上)
  L_cp = 10.0 mm
  L_A = 7.0 mm
  L_B = 14.0 mm
  α = 45°
  θ = 30°
旋转后的点坐标：
  A   = [14.94974747 -2.47487373  9.28660705]
  A+  = [14.94974747  3.58730409 12.78660705]
  A-  = [14.94974747 -8.53705156  5.78660705]
局部坐标系轴方向（在世界坐标中）：
  X' = [1. 0. 0.]
  Y' = [0.        0.8660254 0.5      ]
  Z' = [ 0.        -0.5        0.8660254]
注意：
  - 世界坐标轴为虚线（红X, 绿Y, 蓝Z）
  - 子系统局部坐标轴为实线（红X', 绿Y', 蓝Z'）
  - 旋转轴（青色虚线）经过 CPE，方向沿 X 轴
  - OA 在局部 ZX' 平面内（Y'=0）


In [11]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# 系统参数
# ============================================================

# 世界坐标系中的固定点
z0 = 5.0
L_cp = 10.0
CPS = np.array([0.0, 0.0, z0])
CPE = np.array([L_cp, 0.0, z0])

# rotate-sub-system 内部参数
L_A = 7.0
L_B = 14.0
alpha_deg = 45
theta_deg = 30

# 舵机圆参数（世界坐标，独立于旋转子系统）
x_e = 8.0      # 圆心 X 坐标
y_e = 6.0      # 圆心 Y 坐标（对称）
z_e = 5.0      # 圆心 Z 坐标
R = 4.0        # 圆半径
beta1_deg = 60
beta2_deg = 120

alpha = np.deg2rad(alpha_deg)
theta = np.deg2rad(theta_deg)
beta1 = np.deg2rad(beta1_deg)
beta2 = np.deg2rad(beta2_deg)

# ============================================================
# 辅助函数
# ============================================================

def rot_x_matrix(angle_rad):
    c = np.cos(angle_rad)
    s = np.sin(angle_rad)
    return np.array([
        [1, 0, 0],
        [0, c, -s],
        [0, s, c]
    ])

def transform_to_world(points_local):
    """将局部坐标点（相对于 CPE）变换到世界坐标"""
    Rmat = rot_x_matrix(theta)
    points_world = np.zeros_like(points_local)
    for i, p in enumerate(points_local):
        points_world[i] = CPE + Rmat @ p
    return points_world

def generate_circle_points(center, R, num_points=50):
    """生成圆上的点（世界坐标），圆平行于 zy 平面"""
    phi = np.linspace(0, 2*np.pi, num_points)
    points = np.array([
        center + np.array([0, R*np.cos(phi_i), R*np.sin(phi_i)])
        for phi_i in phi
    ])
    return points

# ============================================================
# 1. 旋转子系统（OA + 垂直线段）
# ============================================================

xA = L_A * np.sin(alpha)
zA = L_A * np.cos(alpha)
A_local = np.array([xA, 0.0, zA])
half_LB = L_B / 2.0
A_plus_local = np.array([xA,  half_LB, zA])
A_minus_local = np.array([xA, -half_LB, zA])

A_world, A_plus_world, A_minus_world = transform_to_world(
    np.array([A_local, A_plus_local, A_minus_local])
)

# ============================================================
# 2. 舵机圆（独立于旋转子系统，世界坐标固定）
# ============================================================

C1 = np.array([x_e,  y_e, z_e])
C2 = np.array([x_e, -y_e, z_e])

P1 = C1 + np.array([0, R*np.cos(beta1), R*np.sin(beta1)])
P2 = C2 + np.array([0, R*np.cos(beta2), R*np.sin(beta2)])

circle1_points = generate_circle_points(C1, R)
circle2_points = generate_circle_points(C2, R)

# ============================================================
# 3. 构建 Plotly 图形
# ============================================================

fig = go.Figure()

# ---- 世界坐标轴（虚线） ----
world_axis_ends = np.array([[15,0,0], [0,15,0], [0,0,15]])
colors = ['red', 'green', 'blue']
labels = ['世界 X', '世界 Y', '世界 Z']
for i, end in enumerate(world_axis_ends):
    fig.add_trace(go.Scatter3d(
        x=[0, end[0]], y=[0, end[1]], z=[0, end[2]],
        mode='lines', line=dict(color=colors[i], width=2, dash='dot'),
        name=labels[i]
    ))

# ---- CPS / CPE ----
fig.add_trace(go.Scatter3d(
    x=[CPS[0], CPE[0]], y=[CPS[1], CPE[1]], z=[CPS[2], CPE[2]],
    mode='markers+text', marker=dict(size=8, color=['green','blue']),
    text=['CPS', 'CPE'], textposition='top center', name='固定点'
))

# ---- cp 线 ----
fig.add_trace(go.Scatter3d(
    x=[CPS[0], CPE[0]], y=[CPS[1], CPE[1]], z=[CPS[2], CPE[2]],
    mode='lines', line=dict(color='gray', width=4, dash='dot'), name='cp 线'
))

# ---- OA 线段 ----
fig.add_trace(go.Scatter3d(
    x=[CPE[0], A_world[0]], y=[CPE[1], A_world[1]], z=[CPE[2], A_world[2]],
    mode='lines', line=dict(color='orange', width=4), name='OA'
))

# ---- 垂直线段 ----
fig.add_trace(go.Scatter3d(
    x=[A_plus_world[0], A_minus_world[0]],
    y=[A_plus_world[1], A_minus_world[1]],
    z=[A_plus_world[2], A_minus_world[2]],
    mode='lines', line=dict(color='purple', width=5), name='垂直线段'
))

# ---- 舵机圆（线框） ----
fig.add_trace(go.Scatter3d(
    x=circle1_points[:,0], y=circle1_points[:,1], z=circle1_points[:,2],
    mode='lines', line=dict(color='cyan', width=2), name='圆 C₁'
))
fig.add_trace(go.Scatter3d(
    x=circle2_points[:,0], y=circle2_points[:,1], z=circle2_points[:,2],
    mode='lines', line=dict(color='magenta', width=2), name='圆 C₂'
))

# ---- 舵机圆心 ----
fig.add_trace(go.Scatter3d(
    x=[C1[0], C2[0]], y=[C1[1], C2[1]], z=[C1[2], C2[2]],
    mode='markers', marker=dict(size=6, color='black'), name='舵机圆心'
))

# ---- 舵机圆上的点（P₁, P₂） ----
fig.add_trace(go.Scatter3d(
    x=[P1[0]], y=[P1[1]], z=[P1[2]],
    mode='markers+text', marker=dict(size=8, color='red'),
    text=['P₁'], textposition='top center', name='舵机点 P₁'
))
fig.add_trace(go.Scatter3d(
    x=[P2[0]], y=[P2[1]], z=[P2[2]],
    mode='markers+text', marker=dict(size=8, color='red'),
    text=['P₂'], textposition='top center', name='舵机点 P₂'
))

# ---- 旋转轴（经过 CPE，沿 X 方向） ----
axis_len = 20
fig.add_trace(go.Scatter3d(
    x=[CPE[0]-axis_len, CPE[0]+axis_len],
    y=[CPE[1], CPE[1]],
    z=[CPE[2], CPE[2]],
    mode='lines', line=dict(color='cyan', width=3, dash='dash'), name='旋转轴'
))

# ============================================================
# 绘图设置
# ============================================================
fig.update_layout(
    title=f'旋转子系统 + 独立舵机圆（相对 ZX 平面对称）<br>'
          f'α={alpha_deg}°, θ={theta_deg}°, β₁={beta1_deg}°, β₂={beta2_deg}°',
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data', camera=dict(eye=dict(x=1.8, y=1.8, z=1.5))
    ),
    width=1000, height=800, showlegend=True
)

fig.show()

# ============================================================
# 打印信息
# ============================================================
print("舵机圆圆心 C₁:", C1)
print("舵机圆圆心 C₂:", C2)
print("P₁ 世界坐标:", P1)
print("P₂ 世界坐标:", P2)

舵机圆圆心 C₁: [8. 6. 5.]
舵机圆圆心 C₂: [ 8. -6.  5.]
P₁ 世界坐标: [8.         8.         8.46410162]
P₂ 世界坐标: [ 8.         -8.          8.46410162]


In [14]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# 系统参数
# ============================================================

# 世界坐标系中的固定点
z0 = 36.0
L_cp = 10.0
CPS = np.array([0.0, 0.0, z0])
CPE = np.array([L_cp, 0.0, z0])

# rotate-sub-system 内部参数
L_A = 7.0         # OA 长度
L_B = 14.0        # 垂直线段长度（A₊ 到 A₋）
alpha_deg = 45    # OA 与局部 +Z 夹角
theta_deg = 30    # 整体旋转角

# 舵机圆参数（世界坐标，独立于旋转子系统）
x_e = 11.0         # 圆心 X 坐标
y_e = 7.0         # 圆心 Y 坐标（对称）
z_e = 0.0         # 圆心 Z 坐标
R = 16.0           # 圆半径
beta1_deg = 60    # 圆 C₁ 上的角度
beta2_deg = 120   # 圆 C₂ 上的角度

alpha = np.deg2rad(alpha_deg)
theta = np.deg2rad(theta_deg)
beta1 = np.deg2rad(beta1_deg)
beta2 = np.deg2rad(beta2_deg)

# ============================================================
# 辅助函数
# ============================================================

def rot_x_matrix(angle_rad):
    c = np.cos(angle_rad)
    s = np.sin(angle_rad)
    return np.array([
        [1, 0, 0],
        [0, c, -s],
        [0, s, c]
    ])

def transform_to_world(points_local):
    """将局部坐标点（相对于 CPE）变换到世界坐标"""
    Rmat = rot_x_matrix(theta)
    points_world = np.zeros_like(points_local)
    for i, p in enumerate(points_local):
        points_world[i] = CPE + Rmat @ p
    return points_world

def generate_circle_points(center, R, num_points=50):
    """生成圆上的点（世界坐标），圆平行于 zy 平面"""
    phi = np.linspace(0, 2*np.pi, num_points)
    points = np.array([
        center + np.array([0, R*np.cos(phi_i), R*np.sin(phi_i)])
        for phi_i in phi
    ])
    return points

# ============================================================
# 1. 旋转子系统（OA + A₊ + A₋）
# ============================================================

# 点 A（局部）
xA = L_A * np.sin(alpha)
zA = L_A * np.cos(alpha)
A_local = np.array([xA, 0.0, zA])

# 垂直线段端点 A₊ 和 A₋（局部）
half_LB = L_B / 2.0
A_plus_local  = np.array([xA,  half_LB, zA])   # A₊
A_minus_local = np.array([xA, -half_LB, zA])   # A₋

# 变换到世界坐标
A_world, A_plus_world, A_minus_world = transform_to_world(
    np.array([A_local, A_plus_local, A_minus_local])
)

# ============================================================
# 2. 舵机圆（独立于旋转子系统，世界坐标固定）
# ============================================================

# 圆心
C1 = np.array([x_e,  y_e, z_e])
C2 = np.array([x_e, -y_e, z_e])

# 圆上的点
P1 = C1 + np.array([0, R*np.cos(beta1), R*np.sin(beta1)])
P2 = C2 + np.array([0, R*np.cos(beta2), R*np.sin(beta2)])

# 完整的圆（用于可视化）
circle1_points = generate_circle_points(C1, R)
circle2_points = generate_circle_points(C2, R)

# ============================================================
# 3. 构建 Plotly 图形
# ============================================================

fig = go.Figure()

# ---- 世界坐标轴（虚线） ----
world_axis_ends = np.array([[15,0,0], [0,15,0], [0,0,15]])
colors = ['red', 'green', 'blue']
labels = ['世界 X', '世界 Y', '世界 Z']
for i, end in enumerate(world_axis_ends):
    fig.add_trace(go.Scatter3d(
        x=[0, end[0]], y=[0, end[1]], z=[0, end[2]],
        mode='lines', line=dict(color=colors[i], width=2, dash='dot'),
        name=labels[i]
    ))

# ---- CPS / CPE ----
fig.add_trace(go.Scatter3d(
    x=[CPS[0], CPE[0]], y=[CPS[1], CPE[1]], z=[CPS[2], CPE[2]],
    mode='markers+text', marker=dict(size=8, color=['green','blue']),
    text=['CPS', 'CPE'], textposition='top center', name='固定点'
))

# ---- cp 线 ----
fig.add_trace(go.Scatter3d(
    x=[CPS[0], CPE[0]], y=[CPS[1], CPE[1]], z=[CPS[2], CPE[2]],
    mode='lines', line=dict(color='gray', width=4, dash='dot'), name='cp 线'
))

# ---- OA 线段（CPE → A） ----
fig.add_trace(go.Scatter3d(
    x=[CPE[0], A_world[0]], y=[CPE[1], A_world[1]], z=[CPE[2], A_world[2]],
    mode='lines', line=dict(color='orange', width=4), name='OA'
))

# ---- 垂直线段（A₊ 到 A₋） ----
fig.add_trace(go.Scatter3d(
    x=[A_plus_world[0], A_minus_world[0]],
    y=[A_plus_world[1], A_minus_world[1]],
    z=[A_plus_world[2], A_minus_world[2]],
    mode='lines', line=dict(color='purple', width=5), name='垂直线段 A₊–A₋'
))

# ---- 点 A₊ 和 A₋ 的标记 ----
fig.add_trace(go.Scatter3d(
    x=[A_plus_world[0], A_minus_world[0]],
    y=[A_plus_world[1], A_minus_world[1]],
    z=[A_plus_world[2], A_minus_world[2]],
    mode='markers+text', marker=dict(size=6, color=['red','red']),
    text=['A₊', 'A₋'], textposition='top center', name='A₊ / A₋'
))

# ---- 点 A 的标记 ----
fig.add_trace(go.Scatter3d(
    x=[A_world[0]], y=[A_world[1]], z=[A_world[2]],
    mode='markers+text', marker=dict(size=6, color='blue'),
    text=['A'], textposition='top center', name='点 A'
))

# ---- 舵机圆（线框） ----
fig.add_trace(go.Scatter3d(
    x=circle1_points[:,0], y=circle1_points[:,1], z=circle1_points[:,2],
    mode='lines', line=dict(color='cyan', width=2), name='舵机圆 C₁'
))
fig.add_trace(go.Scatter3d(
    x=circle2_points[:,0], y=circle2_points[:,1], z=circle2_points[:,2],
    mode='lines', line=dict(color='magenta', width=2), name='舵机圆 C₂'
))

# ---- 舵机圆心 ----
fig.add_trace(go.Scatter3d(
    x=[C1[0], C2[0]], y=[C1[1], C2[1]], z=[C1[2], C2[2]],
    mode='markers', marker=dict(size=6, color='black'), name='舵机圆心'
))

# ---- 舵机圆上的点（P₁, P₂） ----
fig.add_trace(go.Scatter3d(
    x=[P1[0]], y=[P1[1]], z=[P1[2]],
    mode='markers+text', marker=dict(size=8, color='red'),
    text=['P₁'], textposition='top center', name='舵机点 P₁'
))
fig.add_trace(go.Scatter3d(
    x=[P2[0]], y=[P2[1]], z=[P2[2]],
    mode='markers+text', marker=dict(size=8, color='red'),
    text=['P₂'], textposition='top center', name='舵机点 P₂'
))

# ---- 旋转轴（经过 CPE，沿 X 方向） ----
axis_len = 20
fig.add_trace(go.Scatter3d(
    x=[CPE[0]-axis_len, CPE[0]+axis_len],
    y=[CPE[1], CPE[1]],
    z=[CPE[2], CPE[2]],
    mode='lines', line=dict(color='cyan', width=3, dash='dash'), name='旋转轴 (过 CPE)'
))

# ============================================================
# 绘图设置
# ============================================================
fig.update_layout(
    title=f'旋转子系统 (OA + A₊ + A₋) + 独立舵机圆<br>'
          f'α={alpha_deg}°, θ={theta_deg}°, β₁={beta1_deg}°, β₂={beta2_deg}°',
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data', camera=dict(eye=dict(x=1.8, y=1.8, z=1.5))
    ),
    width=1000, height=800, showlegend=True
)

fig.show()

# ============================================================
# 打印信息
# ============================================================
print("=" * 60)
print("旋转子系统（局部 → 世界，θ={}°）：".format(theta_deg))
print(f"  A   = {A_world}")
print(f"  A₊  = {A_plus_world}")
print(f"  A₋  = {A_minus_world}")
print("=" * 60)
print("舵机圆（世界固定，相对于 ZX 平面对称）：")
print(f"  圆心 C₁ = {C1}")
print(f"  圆心 C₂ = {C2}")
print(f"  P₁ (β₁={beta1_deg}°) = {P1}")
print(f"  P₂ (β₂={beta2_deg}°) = {P2}")
print("=" * 60)

旋转子系统（局部 → 世界，θ=30°）：
  A   = [14.94974747 -2.47487373 40.28660705]
  A₊  = [14.94974747  3.58730409 43.78660705]
  A₋  = [14.94974747 -8.53705156 36.78660705]
舵机圆（世界固定，相对于 ZX 平面对称）：
  圆心 C₁ = [11.  7.  0.]
  圆心 C₂ = [11. -7.  0.]
  P₁ (β₁=60°) = [11.         15.         13.85640646]
  P₂ (β₂=120°) = [ 11.         -15.          13.85640646]


In [15]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# 固定参数（所有长度单位：mm）
# ============================================================

# 世界坐标系固定点
z0 = 36.0
L_cp = 10.0
CPS = np.array([0.0, 0.0, z0])
CPE = np.array([L_cp, 0.0, z0])

# 旋转子系统内部固定几何
L_A = 7.0         # CPE → A 的长度
L_B = 14.0        # A₊ 到 A₋ 的垂直距离

# 舵机圆固定参数
x_e = 11.0
y_e = 7.0
z_e = 0.0
R = 16.0

# ============================================================
# 初始状态的角度（可变参数）
# ============================================================
alpha_deg = 45.0   # CPE→A 与局部 +Z 的夹角
theta_deg = 0.0    # 子系统绕 X 轴（过 CPE）的旋转角
beta1_deg = 60.0   # 右舵机角度（C₁）
beta2_deg = 120.0  # 左舵机角度（C₂）

alpha = np.deg2rad(alpha_deg)
theta = np.deg2rad(theta_deg)
beta1 = np.deg2rad(beta1_deg)
beta2 = np.deg2rad(beta2_deg)

# ============================================================
# 辅助函数
# ============================================================

def rot_x_matrix(angle_rad):
    c = np.cos(angle_rad)
    s = np.sin(angle_rad)
    return np.array([
        [1, 0, 0],
        [0, c, -s],
        [0, s, c]
    ])

def transform_to_world(points_local):
    """将局部坐标点（相对于 CPE）变换到世界坐标"""
    Rmat = rot_x_matrix(theta)
    points_world = np.zeros_like(points_local)
    for i, p in enumerate(points_local):
        points_world[i] = CPE + Rmat @ p
    return points_world

def generate_circle_points(center, R, num_points=50):
    phi = np.linspace(0, 2*np.pi, num_points)
    points = np.array([
        center + np.array([0, R*np.cos(phi_i), R*np.sin(phi_i)])
        for phi_i in phi
    ])
    return points

# ============================================================
# 1. 旋转子系统：A, A₊, A₋
# ============================================================

# 点 A（局部坐标）
xA = L_A * np.sin(alpha)
zA = L_A * np.cos(alpha)
A_local = np.array([xA, 0.0, zA])

# 垂直线段端点（局部坐标）
half_LB = L_B / 2.0
A_plus_local  = np.array([xA,  half_LB, zA])
A_minus_local = np.array([xA, -half_LB, zA])

# 变换到世界坐标
A_world, A_plus_world, A_minus_world = transform_to_world(
    np.array([A_local, A_plus_local, A_minus_local])
)

# ============================================================
# 2. 舵机圆（世界固定）
# ============================================================

C1 = np.array([x_e,  y_e, z_e])
C2 = np.array([x_e, -y_e, z_e])

P1 = C1 + np.array([0, R*np.cos(beta1), R*np.sin(beta1)])
P2 = C2 + np.array([0, R*np.cos(beta2), R*np.sin(beta2)])

circle1_points = generate_circle_points(C1, R)
circle2_points = generate_circle_points(C2, R)

# ============================================================
# 3. 计算左右连杆长度（初始状态应相等）
# ============================================================
link_R = np.linalg.norm(A_plus_world - P1)
link_L = np.linalg.norm(A_minus_world - P2)
link_diff = link_R - link_L

# ============================================================
# 4. 可视化
# ============================================================

fig = go.Figure()

# ---- 世界坐标轴 ----
world_axis_ends = np.array([[30,0,0], [0,30,0], [0,0,50]])
colors = ['red', 'green', 'blue']
labels = ['世界 X', '世界 Y', '世界 Z']
for i, end in enumerate(world_axis_ends):
    fig.add_trace(go.Scatter3d(
        x=[0, end[0]], y=[0, end[1]], z=[0, end[2]],
        mode='lines', line=dict(color=colors[i], width=2, dash='dot'),
        name=labels[i]
    ))

# ---- CPS / CPE ----
fig.add_trace(go.Scatter3d(
    x=[CPS[0], CPE[0]], y=[CPS[1], CPE[1]], z=[CPS[2], CPE[2]],
    mode='markers+text', marker=dict(size=8, color=['green','blue']),
    text=['CPS', 'CPE'], textposition='top center', name='固定点'
))

# ---- cp 线 ----
fig.add_trace(go.Scatter3d(
    x=[CPS[0], CPE[0]], y=[CPS[1], CPE[1]], z=[CPS[2], CPE[2]],
    mode='lines', line=dict(color='gray', width=4, dash='dot'), name='cp 线'
))

# ---- CPE → A ----
fig.add_trace(go.Scatter3d(
    x=[CPE[0], A_world[0]], y=[CPE[1], A_world[1]], z=[CPE[2], A_world[2]],
    mode='lines', line=dict(color='orange', width=4), name='CPE → A'
))

# ---- A₊ 到 A₋ 垂直线段 ----
fig.add_trace(go.Scatter3d(
    x=[A_plus_world[0], A_minus_world[0]],
    y=[A_plus_world[1], A_minus_world[1]],
    z=[A_plus_world[2], A_minus_world[2]],
    mode='lines', line=dict(color='purple', width=5), name='垂直线段 A₊–A₋'
))

# ---- 连杆（A₊ → P₁ 和 A₋ → P₂） ----
fig.add_trace(go.Scatter3d(
    x=[A_plus_world[0], P1[0]], y=[A_plus_world[1], P1[1]], z=[A_plus_world[2], P1[2]],
    mode='lines', line=dict(color='red', width=3), name=f'右连杆 ({link_R:.2f} mm)'
))
fig.add_trace(go.Scatter3d(
    x=[A_minus_world[0], P2[0]], y=[A_minus_world[1], P2[1]], z=[A_minus_world[2], P2[2]],
    mode='lines', line=dict(color='blue', width=3), name=f'左连杆 ({link_L:.2f} mm)'
))

# ---- 关键点标记 ----
fig.add_trace(go.Scatter3d(
    x=[A_world[0], A_plus_world[0], A_minus_world[0]],
    y=[A_world[1], A_plus_world[1], A_minus_world[1]],
    z=[A_world[2], A_plus_world[2], A_minus_world[2]],
    mode='markers+text', marker=dict(size=6, color=['blue', 'red', 'red']),
    text=['A', 'A₊', 'A₋'], textposition='top center', name='子系统点'
))

# ---- 舵机圆 ----
fig.add_trace(go.Scatter3d(
    x=circle1_points[:,0], y=circle1_points[:,1], z=circle1_points[:,2],
    mode='lines', line=dict(color='cyan', width=2), name='舵机圆 C₁'
))
fig.add_trace(go.Scatter3d(
    x=circle2_points[:,0], y=circle2_points[:,1], z=circle2_points[:,2],
    mode='lines', line=dict(color='magenta', width=2), name='舵机圆 C₂'
))

# ---- 舵机圆心 ----
fig.add_trace(go.Scatter3d(
    x=[C1[0], C2[0]], y=[C1[1], C2[1]], z=[C1[2], C2[2]],
    mode='markers', marker=dict(size=6, color='black'), name='舵机圆心'
))

# ---- 舵机点 P₁, P₂ ----
fig.add_trace(go.Scatter3d(
    x=[P1[0]], y=[P1[1]], z=[P1[2]],
    mode='markers+text', marker=dict(size=8, color='red'),
    text=['P₁'], textposition='top center', name='舵机点 P₁'
))
fig.add_trace(go.Scatter3d(
    x=[P2[0]], y=[P2[1]], z=[P2[2]],
    mode='markers+text', marker=dict(size=8, color='red'),
    text=['P₂'], textposition='top center', name='舵机点 P₂'
))

# ---- 旋转轴（过 CPE，沿 X 轴） ----
axis_len = 30
fig.add_trace(go.Scatter3d(
    x=[CPE[0]-axis_len, CPE[0]+axis_len],
    y=[CPE[1], CPE[1]],
    z=[CPE[2], CPE[2]],
    mode='lines', line=dict(color='cyan', width=3, dash='dash'), name='旋转轴 (过 CPE)'
))

# ============================================================
# 图形设置
# ============================================================
fig.update_layout(
    title=f'初始状态 | α={alpha_deg}°, θ={theta_deg}°, β₁={beta1_deg}°, β₂={beta2_deg}°<br>'
          f'右连杆长度 = {link_R:.2f} mm, 左连杆长度 = {link_L:.2f} mm, 差值 = {link_diff:.2e} mm',
    scene=dict(
        xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
        aspectmode='data', camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
    ),
    width=1000, height=800, showlegend=True
)

fig.show()

# ============================================================
# 打印信息
# ============================================================
print("=" * 60)
print("初始状态参数：")
print(f"  α = {alpha_deg}°, θ = {theta_deg}°")
print(f"  β₁ = {beta1_deg}°, β₂ = {beta2_deg}°")
print(f"  CPE→A 长度 = {L_A} mm")
print(f"  A₊–A₋ 长度 = {L_B} mm")
print("=" * 60)
print("关键点坐标（世界）：")
print(f"  A   = {A_world}")
print(f"  A₊  = {A_plus_world}")
print(f"  A₋  = {A_minus_world}")
print(f"  P₁ = {P1}")
print(f"  P₂ = {P2}")
print("=" * 60)
print(f"连杆长度：")
print(f"  右侧 (A₊→P₁) = {link_R:.4f} mm")
print(f"  左侧 (A₋→P₂) = {link_L:.4f} mm")
print(f"  差值 = {link_diff:.2e} mm (应接近 0)")
print("=" * 60)

初始状态参数：
  α = 45.0°, θ = 0.0°
  β₁ = 60.0°, β₂ = 120.0°
  CPE→A 长度 = 7.0 mm
  A₊–A₋ 长度 = 14.0 mm
关键点坐标（世界）：
  A   = [14.94974747  0.         40.94974747]
  A₊  = [14.94974747  7.         40.94974747]
  A₋  = [14.94974747 -7.         40.94974747]
  P₁ = [11.         15.         13.85640646]
  P₂ = [ 11.         -15.          13.85640646]
连杆长度：
  右侧 (A₊→P₁) = 28.5245 mm
  左侧 (A₋→P₂) = 28.5245 mm
  差值 = 7.11e-15 mm (应接近 0)


In [17]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import fsolve
from ipywidgets import interact, FloatSlider
from IPython.display import display

# ============================================================
# 固定参数（所有长度单位：mm）
# ============================================================
z0 = 36.0
L_cp = 10.0
CPS = np.array([0.0, 0.0, z0])
CPE = np.array([L_cp, 0.0, z0])

L_A = 7.0
L_B = 14.0

x_e = 11.0
y_e = 7.0
z_e = 0.0
R = 16.0

# 固定连杆长度（从初始状态计算得到）
L_link = 28.4  # 你可以微调，但必须使系统有解

# ============================================================
# 辅助函数
# ============================================================
def rot_x_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1,0,0], [0,c,-s], [0,s,c]])

def A_plus_global(alpha, theta):
    """返回 A₊ 的世界坐标"""
    xA = L_A * np.sin(alpha)
    zA = L_A * np.cos(alpha)
    A_plus_local = np.array([xA, L_B/2, zA])
    Rmat = rot_x_matrix(theta)
    return CPE + Rmat @ A_plus_local

def A_minus_global(alpha, theta):
    """返回 A₋ 的世界坐标"""
    xA = L_A * np.sin(alpha)
    zA = L_A * np.cos(alpha)
    A_minus_local = np.array([xA, -L_B/2, zA])
    Rmat = rot_x_matrix(theta)
    return CPE + Rmat @ A_minus_local

def P1_global(beta1):
    """舵机点 P₁ 的世界坐标"""
    return np.array([x_e, y_e + R*np.cos(beta1), z_e + R*np.sin(beta1)])

def P2_global(beta2):
    """舵机点 P₂ 的世界坐标"""
    return np.array([x_e, -y_e + R*np.cos(beta2), z_e + R*np.sin(beta2)])

def equations(vars, beta1, beta2):
    """方程组：两个连杆长度等于 L_link"""
    alpha, theta = vars
    A_plus = A_plus_global(alpha, theta)
    A_minus = A_minus_global(alpha, theta)
    P1 = P1_global(beta1)
    P2 = P2_global(beta2)
    eq1 = np.linalg.norm(A_plus - P1) - L_link
    eq2 = np.linalg.norm(A_minus - P2) - L_link
    return [eq1, eq2]

def solve_alpha_theta(beta1, beta2, guess=(0.8, 0.0)):
    """求解 alpha, theta (弧度)"""
    sol = fsolve(equations, guess, args=(beta1, beta2), xtol=1e-8)
    return sol[0], sol[1]

# ============================================================
# 可视化函数
# ============================================================
def generate_circle_points(center, R, num=50):
    phi = np.linspace(0, 2*np.pi, num)
    return np.array([center + [0, R*np.cos(p), R*np.sin(p)] for p in phi])

def update_plot(beta1_deg, beta2_deg):
    beta1 = np.deg2rad(beta1_deg)
    beta2 = np.deg2rad(beta2_deg)

    # 求解 alpha, theta
    try:
        alpha, theta = solve_alpha_theta(beta1, beta2)
        alpha_deg = np.rad2deg(alpha)
        theta_deg = np.rad2deg(theta)
    except:
        print(f"警告: 无解 for β₁={beta1_deg}°, β₂={beta2_deg}°")
        return

    # 计算关键点
    A_plus = A_plus_global(alpha, theta)
    A_minus = A_minus_global(alpha, theta)
    P1 = P1_global(beta1)
    P2 = P2_global(beta2)

    # 圆上的点（用于可视化）
    C1 = np.array([x_e,  y_e, z_e])
    C2 = np.array([x_e, -y_e, z_e])
    circ1 = generate_circle_points(C1, R)
    circ2 = generate_circle_points(C2, R)

    # 连杆长度（验证）
    lenR = np.linalg.norm(A_plus - P1)
    lenL = np.linalg.norm(A_minus - P2)

    # 创建图形
    fig = go.Figure()

    # 世界坐标轴
    axes = [[30,0,0], [0,30,0], [0,0,50]]
    colors = ['red','green','blue']
    labels = ['X','Y','Z']
    for i, end in enumerate(axes):
        fig.add_trace(go.Scatter3d(
            x=[0,end[0]], y=[0,end[1]], z=[0,end[2]],
            mode='lines', line=dict(color=colors[i], width=2, dash='dot'),
            name=f'世界{labels[i]}'
        ))

    # CPS, CPE, cp线
    fig.add_trace(go.Scatter3d(
        x=[CPS[0], CPE[0]], y=[CPS[1], CPE[1]], z=[CPS[2], CPE[2]],
        mode='markers+text', marker=dict(size=8, color=['green','blue']),
        text=['CPS','CPE'], textposition='top center', name='固定点'
    ))
    fig.add_trace(go.Scatter3d(
        x=[CPS[0], CPE[0]], y=[CPS[1], CPE[1]], z=[CPS[2], CPE[2]],
        mode='lines', line=dict(color='gray', width=3, dash='dot'), name='cp 线'
    ))

    # 子系统: 垂直线段 A₊–A₋
    fig.add_trace(go.Scatter3d(
        x=[A_plus[0], A_minus[0]],
        y=[A_plus[1], A_minus[1]],
        z=[A_plus[2], A_minus[2]],
        mode='lines', line=dict(color='purple', width=5), name='A₊–A₋'
    ))

    # 连杆
    fig.add_trace(go.Scatter3d(
        x=[A_plus[0], P1[0]], y=[A_plus[1], P1[1]], z=[A_plus[2], P1[2]],
        mode='lines', line=dict(color='red', width=3),
        name=f'右连杆 ({lenR:.2f} mm)'
    ))
    fig.add_trace(go.Scatter3d(
        x=[A_minus[0], P2[0]], y=[A_minus[1], P2[1]], z=[A_minus[2], P2[2]],
        mode='lines', line=dict(color='blue', width=3),
        name=f'左连杆 ({lenL:.2f} mm)'
    ))

    # 关键点标记
    fig.add_trace(go.Scatter3d(
        x=[A_plus[0], A_minus[0]], y=[A_plus[1], A_minus[1]], z=[A_plus[2], A_minus[2]],
        mode='markers+text', marker=dict(size=6, color=['red','red']),
        text=['A₊','A₋'], textposition='top center', name='子系统端点'
    ))

    # 舵机圆
    fig.add_trace(go.Scatter3d(
        x=circ1[:,0], y=circ1[:,1], z=circ1[:,2],
        mode='lines', line=dict(color='cyan', width=2), name='舵机圆 C₁'
    ))
    fig.add_trace(go.Scatter3d(
        x=circ2[:,0], y=circ2[:,1], z=circ2[:,2],
        mode='lines', line=dict(color='magenta', width=2), name='舵机圆 C₂'
    ))

    # 圆心
    fig.add_trace(go.Scatter3d(
        x=[C1[0], C2[0]], y=[C1[1], C2[1]], z=[C1[2], C2[2]],
        mode='markers', marker=dict(size=6, color='black'), name='圆心'
    ))

    # 舵机点
    fig.add_trace(go.Scatter3d(
        x=[P1[0]], y=[P1[1]], z=[P1[2]],
        mode='markers+text', marker=dict(size=8, color='red'),
        text=['P₁'], textposition='top center', name='舵机点 P₁'
    ))
    fig.add_trace(go.Scatter3d(
        x=[P2[0]], y=[P2[1]], z=[P2[2]],
        mode='markers+text', marker=dict(size=8, color='red'),
        text=['P₂'], textposition='top center', name='舵机点 P₂'
    ))

    # 旋转轴
    axis_len = 30
    fig.add_trace(go.Scatter3d(
        x=[CPE[0]-axis_len, CPE[0]+axis_len],
        y=[CPE[1], CPE[1]],
        z=[CPE[2], CPE[2]],
        mode='lines', line=dict(color='cyan', width=3, dash='dash'),
        name='旋转轴 (过 CPE)'
    ))

    # 布局
    fig.update_layout(
        title=f'β₁={beta1_deg}°, β₂={beta2_deg}° → α={alpha_deg:.2f}°, θ={theta_deg:.2f}°<br>'
              f'右连杆={lenR:.2f} mm, 左连杆={lenL:.2f} mm, 目标 L_link={L_link} mm',
        scene=dict(
            xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
            aspectmode='data', camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
        ),
        width=1000, height=800, showlegend=True
    )

    fig.show()

# ============================================================
# 交互控件
# ============================================================
interact(
    update_plot,
    beta1_deg=FloatSlider(min=0, max=360, step=1, value=60, description='β₁ (deg)'),
    beta2_deg=FloatSlider(min=0, max=360, step=1, value=120, description='β₂ (deg)')
);

interactive(children=(FloatSlider(value=60.0, description='β₁ (deg)', max=360.0, step=1.0), FloatSlider(value=…